<div class="alert alert-block alert-success" style="font-family: Times New Roman">
    <h4><strong>Laboratory Task 5</strong></h4>

**1. Perform Standard Imports**

<br>

**2. Create a function called `set_seed()` that accepts `seed: int` as a parameter, this function must return nothing but just set the seed to a certain value.**

<br>

**3. Create a NumPy array called "arr" that contains 6 random integers between 0 (inclusive) and 5 (exclusive), call the `set_seed()` function and use `42` as the seed parameter.**

<br>

**4. Create a tensor "x" from the array above**

<br>

**5. Change the dtype of x from `int32` to `int64`**

<br>

**6. Reshape `x` into a 3x2 tensor** <br> There are several ways to do this.

<br>

**7. Return the right-hand column of tensor `x`**

<br>

**8. Without changing x, return a tensor of square values of `x`** <br> There are several ways to do this.

<br>

**9. Create a tensor `y` with the same number of elements as `x`, that can be matrix-multiplied with `x`** <br> Use PyTorch directly (not NumPy) to create a tensor of random integers between 0 (inclusive) and 5 (exclusive). Use 42 as seed. <br> Think about what shape it should have to permit matrix multiplication.

<br>

**10. Find the matrix product of `x` and `y`.**

</div>

### My Approach

This task chains together most of the tensor fundamentals from the lecture notebook — creation from NumPy, dtypes, reshaping, indexing, element-wise math, and matrix multiplication — into one worked example. I'll go step by step in the exact order given, explaining the reasoning behind each choice (especially wherever "there are several ways to do this" is called out, since picking one deliberately and knowing the alternatives is the actual point of those steps).

A few things I want to be careful about before writing any code:

- **Reproducibility across two different random generators.** Step 3 needs a *NumPy* random array, and step 9 needs a *PyTorch* random tensor — both seeded with 42. NumPy and PyTorch keep their own, independent random number generators, so a single seeding call has to cover both if `set_seed()` is going to make *any* random operation in this notebook reproducible, regardless of which library generates it. So `set_seed()` will seed both `numpy.random` and `torch`.
- **Platform-dependent default integer dtype.** `np.random.randint()` returns platform-dependent integers — on Windows this defaults to 32-bit (`int32`), but on Linux/macOS it's usually 64-bit (`int64`). Step 5 explicitly frames the task as "change the dtype of x from `int32` to `int64`", so to make this notebook behave the same way regardless of which OS it's run on, I explicitly cast `arr` to `int32` right after creating it. That way step 5's conversion to `int64` is a real, visible dtype change here too, not a no-op.
- **Shape of `y` for matrix multiplication.** `x` ends up as a $3\times2$ tensor (6 elements). For $x \times y$ to be valid matrix multiplication, $y$'s number of *rows* must equal $x$'s number of *columns* — i.e. $y$ must have 2 rows. Combined with the requirement that $y$ has the *same number of elements as* `x` (6), that pins down $y$'s shape uniquely: $2 \times 3$ ($2 \times 3 = 6$). This also conveniently produces a square $3\times3$ result for $x \times y$, which is a nice, easy shape to sanity-check by eye.

With that groundwork laid out, here's the notebook.

### 1. Perform Standard Imports

In [1]:
import torch
import numpy as np

### 2. `set_seed()` function

As discussed above, this needs to seed **both** NumPy's and PyTorch's random number generators, since the notebook uses both `np.random` (step 3) and `torch` (step 9) for randomness. The function returns nothing — it just has the *side effect* of putting both generators into a reproducible state.

In [2]:
# Seed every random number generator used in this notebook so that every
# subsequent "random" operation (NumPy or PyTorch) is reproducible.
#
# Parameters
# ----------
# seed : int
#     The seed value to use for both NumPy's and PyTorch's RNGs.
#
# Returns
# -------
# None
#     This function has no return value -- it only has the side effect
#     of setting global RNG state.
def set_seed(seed: int):
    np.random.seed(seed)
    torch.manual_seed(seed)


# quick smoke test: calling it shouldn't raise, and shouldn't return anything
return_value = set_seed(42)
print("set_seed(42) returned:", return_value)

set_seed(42) returned: None


### 3. Create the NumPy array `arr`

`np.random.randint(low, high, size)` draws `size` integers from the half-open interval `[low, high)` — so `low=0, high=5` gives integers from `{0, 1, 2, 3, 4}`, matching "between 0 (inclusive) and 5 (exclusive)". I call `set_seed(42)` immediately beforehand so the exact 6 numbers drawn are reproducible.

As noted above, I then explicitly cast the result to `int32`. This isn't strictly required by NumPy (which might already hand back `int64` depending on the OS), but it guarantees the very next step — converting `int32` to `int64` — is meaningful on any machine that runs this notebook.

In [3]:
set_seed(42)
arr = np.random.randint(0, 5, 6).astype(np.int32)

print("arr      =", arr)
print("dtype    =", arr.dtype)
print("shape    =", arr.shape)

arr      = [3 4 2 4 4 1]
dtype    = int32
shape    = (6,)


### 4. Create a tensor `x` from `arr`

The lecture notebook introduced three ways to build a tensor from a NumPy array: `torch.from_numpy()`, `torch.as_tensor()` (both of which *share memory* with the original array), and `torch.tensor()` (which always copies). Since I don't need `arr` and `x` to stay linked here — and copying is the safer default when there's no specific reason to share memory — I use `torch.from_numpy()` mainly to keep this consistent with how the lecture introduced NumPy→tensor conversion, while noting the distinction.

In [4]:
x = torch.from_numpy(arr)

print(x)
print("dtype:", x.dtype)
print("tensor type:", x.type())

tensor([3, 4, 2, 4, 4, 1], dtype=torch.int32)
dtype: torch.int32
tensor type: torch.IntTensor


### 5. Change the dtype of `x` from `int32` to `int64`

The lecture notebook is explicit about this: don't write `x = torch.tensor(x, dtype=torch.int64)` (that raises a warning/error about improper tensor cloning). Instead, use the tensor's own `.type()` method, which returns a new tensor with the requested dtype.

In [5]:
print("Before:", x.dtype)

x = x.type(torch.int64)

print("After: ", x.dtype)
print(x)

Before: torch.int32
After:  torch.int64
tensor([3, 4, 2, 4, 4, 1])


### 6. Reshape `x` into a 3x2 tensor

`arr`/`x` has 6 elements, and $3 \times 2 = 6$, so this reshape is valid. PyTorch offers a couple of essentially equivalent ways to do this:

- **`.view(3, 2)`** — returns a new tensor that is a *view* on the same underlying memory (fast, no copy, but only works when the data is contiguous in memory).
- **`.reshape(3, 2)`** — returns a view when possible, and transparently falls back to making a copy when a view isn't possible (e.g. after certain operations that leave the data non-contiguous). It's the more "forgiving" option.

Either works here since `x`'s data is contiguous, so I'll demonstrate both and confirm they give identical results before settling on one for the rest of the notebook.

In [6]:
x_view = x.view(3, 2)
x_reshape = x.reshape(3, 2)

print("x.view(3,2)    =\n", x_view)
print("x.reshape(3,2) =\n", x_reshape)
print("Same values?", torch.equal(x_view, x_reshape))

# Commit to one going forward -- reshape() is the safer general-purpose choice
x = x.reshape(3, 2)
print("\nx is now:\n", x)
print("x.shape =", x.shape)

x.view(3,2)    =
 tensor([[3, 4],
        [2, 4],
        [4, 1]])
x.reshape(3,2) =
 tensor([[3, 4],
        [2, 4],
        [4, 1]])
Same values? True

x is now:
 tensor([[3, 4],
        [2, 4],
        [4, 1]])
x.shape = torch.Size([3, 2])


### 7. Return the right-hand column of `x`

With `x` shaped `(3, 2)`, "the right-hand column" is column index `1` (the last of the two columns). Using the slicing syntax `x[:, 1]` means "every row (`:`), column `1`" — exactly matching the indexing example already shown in the lecture notebook for a differently-valued `(3,2)` tensor.

In [7]:
right_col = x[:, 1]
print("Right-hand column x[:, 1] =", right_col)

# Note: slicing with 1: instead of a bare 1 keeps it as a column vector (3,1)
# rather than flattening it to a 1D tensor of shape (3,) -- useful to know,
# even though the task just asks for "the right-hand column" (either is valid).
right_col_as_column = x[:, 1:]
print("\nAs a (3,1) column tensor:\n", right_col_as_column)

Right-hand column x[:, 1] = tensor([4, 4, 1])

As a (3,1) column tensor:
 tensor([[4],
        [4],
        [1]])


### 8. Square values of `x`, without changing `x`

"Several ways to do this" — all of the following compute the element-wise square **without modifying `x` in place** (none of them use an in-place, underscore-suffixed method like `.pow_()`, and none of them use the `*=`/`**=` in-place operators):

1. `x ** 2` — Python's built-in power operator, works element-wise on tensors.
2. `x.pow(2)` — the tensor method equivalent of `torch.pow(x, 2)`.
3. `torch.square(x)` — a dedicated PyTorch function for squaring.
4. `x * x` — plain element-wise multiplication of `x` with itself.

I compute all four and confirm they agree with each other, and then confirm `x` itself is untouched.

In [8]:
squared_pow_operator = x ** 2
squared_pow_method = x.pow(2)
squared_torch_square = torch.square(x)
squared_self_multiply = x * x

print("x ** 2         =\n", squared_pow_operator)
print("x.pow(2)       =\n", squared_pow_method)
print("torch.square(x)=\n", squared_torch_square)
print("x * x          =\n", squared_self_multiply)

all_agree = (
    torch.equal(squared_pow_operator, squared_pow_method)
    and torch.equal(squared_pow_method, squared_torch_square)
    and torch.equal(squared_torch_square, squared_self_multiply)
)
print("\nAll four methods agree:", all_agree)

print("\nx is unchanged:\n", x)

x ** 2         =
 tensor([[ 9, 16],
        [ 4, 16],
        [16,  1]])
x.pow(2)       =
 tensor([[ 9, 16],
        [ 4, 16],
        [16,  1]])
torch.square(x)=
 tensor([[ 9, 16],
        [ 4, 16],
        [16,  1]])
x * x          =
 tensor([[ 9, 16],
        [ 4, 16],
        [16,  1]])

All four methods agree: True

x is unchanged:
 tensor([[3, 4],
        [2, 4],
        [4, 1]])


### 9. Create tensor `y`, matrix-multiplication-compatible with `x`

As reasoned above: `x` is $3\times2$, and matrix multiplication $x \times y$ requires the number of columns of `x` (2) to equal the number of rows of `y`. Combined with "the same number of elements as `x`" (6 elements total), `y` must be shaped $2\times3$ ($2 \times 3 = 6$).

This time the randomness must come from **PyTorch directly** (`torch.randint`), not NumPy — so I call `set_seed(42)` again immediately before it, which reseeds `torch`'s generator (as well as NumPy's, which is harmless here since I'm not drawing any more NumPy randomness).

In [9]:
set_seed(42)
y = torch.randint(0, 5, (2, 3))

print("y =\n", y)
print("y.shape =", y.shape, " (x.shape =", x.shape, ")")
print("y has the same number of elements as x:", y.numel() == x.numel())
print("Inner dimensions match for x @ y: x.shape[1] =", x.shape[1], " y.shape[0] =", y.shape[0])

y =
 tensor([[2, 2, 1],
        [4, 1, 0]])
y.shape = torch.Size([2, 3])  (x.shape = torch.Size([3, 2]) )
y has the same number of elements as x: True
Inner dimensions match for x @ y: x.shape[1] = 2  y.shape[0] = 2


### 10. Matrix product of `x` and `y`

As shown in the lecture notebook, 2D matrix multiplication can be written as `torch.mm(a, b)`, `a.mm(b)`, or simply `a @ b` — all three are equivalent for plain 2D tensors like these (no broadcasting is needed here, so `torch.matmul`/`@` and `torch.mm` behave identically). The result of a $(3\times2) \times (2\times3)$ multiplication is a $(3\times3)$ tensor.

In [10]:
product_mm = torch.mm(x, y)
product_method = x.mm(y)
product_at = x @ y

print("torch.mm(x, y) =\n", product_mm)
print("\nx.mm(y)        =\n", product_method)
print("\nx @ y          =\n", product_at)

print("\nAll three agree:", torch.equal(product_mm, product_method) and torch.equal(product_method, product_at))
print("Result shape:", product_mm.shape, " (expected (3,3) from a (3,2) x (2,3) multiplication)")

torch.mm(x, y) =
 tensor([[22, 10,  3],
        [20,  8,  2],
        [12,  9,  4]])

x.mm(y)        =
 tensor([[22, 10,  3],
        [20,  8,  2],
        [12,  9,  4]])

x @ y          =
 tensor([[22, 10,  3],
        [20,  8,  2],
        [12,  9,  4]])

All three agree: True
Result shape: torch.Size([3, 3])  (expected (3,3) from a (3,2) x (2,3) multiplication)


### Summary

- `set_seed()` seeds **both** NumPy's and PyTorch's random number generators in one call, which mattered because this task draws randomness from each library at a different point (`np.random.randint` for `arr`, `torch.randint` for `y`) and both needed to be reproducible with the same seed value, `42`.
- Converting between NumPy and PyTorch, then between dtypes, showed why the lecture notebook warns against `torch.tensor(x, dtype=...)` for changing an existing tensor's dtype — `.type()` is the correct tool for that job.
- `.view()` and `.reshape()` produced identical results for reshaping `x` into $3\times2$, since the underlying data was contiguous; `.reshape()` is the more robust default to reach for in general, since it degrades gracefully to a copy when a view isn't possible.
- Indexing with `x[:, 1]` pulled out the right-hand column as a 1D tensor, while `x[:, 1:]` gives the same values as a $(3,1)$ column — both are legitimate answers depending on whether a flattened or a column-shaped result is wanted downstream.
- Squaring `x` four different ways (`**`, `.pow()`, `torch.square()`, self-multiplication) all agreed and left `x` itself untouched, since none of them were in-place operations.
- Getting `y`'s shape right came down to two constraints working together: "same number of elements as `x`" (6) and "matrix-multiplication-compatible with `x`" (needs 2 rows to match `x`'s 2 columns) — together these forced `y` to be exactly $2\times3$, with no other shape satisfying both.
- `torch.mm(x, y)`, `x.mm(y)`, and `x @ y` all produced the same $3\times3$ product, consistent with the $(3,2) \times (2,3) \rightarrow (3,3)$ shape rule for matrix multiplication.